In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from IPython.display import HTML

np.random.seed(42)

n_normal = 320
x1 = np.random.normal(50, 8, n_normal)
x2 = np.random.normal(30, 5, n_normal)

n_global = 18
x1_g = np.random.normal(92, 4, n_global)
x2_g = np.random.normal(60, 4, n_global)

n_local = 18
x1_l = np.random.normal(35, 2.2, n_local)
x2_l = np.random.normal(48, 1.6, n_local)

X = np.vstack([
    np.column_stack([x1, x2]),
    np.column_stack([x1_g, x2_g]),
    np.column_stack([x1_l, x2_l])
])

idx = np.random.permutation(len(X))
X = X[idx]

q1_x = np.percentile(X[:, 0], 25)
q3_x = np.percentile(X[:, 0], 75)
iqr_x = q3_x - q1_x
lower_x = q1_x - 1.5 * iqr_x
upper_x = q3_x + 1.5 * iqr_x

q1_y = np.percentile(X[:, 1], 25)
q3_y = np.percentile(X[:, 1], 75)
iqr_y = q3_y - q1_y
lower_y = q1_y - 1.5 * iqr_y
upper_y = q3_y + 1.5 * iqr_y

iqr_flag = (
    (X[:, 0] < lower_x) | (X[:, 0] > upper_x) |
    (X[:, 1] < lower_y) | (X[:, 1] > upper_y)
)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.08)
lof_pred = lof.fit_predict(X)
lof_flag = lof_pred == -1

iso = IsolationForest(
    n_estimators=200,
    contamination=0.08,
    random_state=42
)
iso_pred = iso.fit_predict(X)
iso_flag = iso_pred == -1

fig, axes = plt.subplots(1, 3, figsize=(13, 4.9), facecolor="white")
plt.subplots_adjust(top=0.80, bottom=0.20, wspace=0.24)

titles = [
    "IQR",
    "Local Outlier Factor",
    "Isolation Forest"
]

flags = [iqr_flag, lof_flag, iso_flag]

main_color = "#53c5ff"
outlier_color = "#ff3131"

x_min, x_max = X[:, 0].min() - 5, X[:, 0].max() + 5
y_min, y_max = X[:, 1].min() - 5, X[:, 1].max() + 5

total_frames = 80

def draw_frame(frame):
    for ax in axes:
        ax.clear()

    fig.suptitle(
        "Outliers: diferentes métodos, diferentes leituras",
        fontsize=16,
        fontweight="bold",
        y=0.95
    )

    frac_pts = min(1, (frame + 1) / 32)
    n_pts = max(12, int(len(X) * frac_pts))

    frac_out = min(1, max(0, (frame - 18) / 20))
    n_out = max(0, int(n_pts * frac_out))

    for ax, title, flag in zip(axes, titles, flags):
        current_X = X[:n_pts]
        current_flag = flag[:n_pts]

        ax.scatter(
            current_X[~current_flag, 0],
            current_X[~current_flag, 1],
            s=24,
            alpha=0.65,
            color=main_color
        )

        out_idx = np.where(current_flag)[0]
        out_idx = out_idx[:n_out] if len(out_idx) > 0 else []

        if len(out_idx) > 0:
            ax.scatter(
                current_X[out_idx, 0],
                current_X[out_idx, 1],
                s=38,
                alpha=0.95,
                color=outlier_color
            )

        ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel("Variável 1", fontsize=9)
        ax.set_ylabel("Variável 2", fontsize=9)
        ax.grid(alpha=0.16)

        ax.text(
            0.03, 0.95,
            f"Outliers sinalizados: {int(current_flag.sum()) if frame > 35 else min(int(current_flag.sum()), len(out_idx))}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9
        )

    if frame >= 52:
        fig.text(
            0.5, 0.06,
            "IQR captura extremos univariados; LOF e Isolation Forest incorporam estrutura multivariada.",
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

anim = FuncAnimation(fig, draw_frame, frames=total_frames, interval=220, repeat=True)

HTML(anim.to_jshtml())